# ASR Preset

Reusable Google Colab notebook template for automatic speech recognition competitions and demos.

This preset is based on a Typhoon ASR real-time inference notebook, but makes the pipeline configurable:

- Install and load ASR dependencies
- Mount Drive or upload/extract a dataset zip
- Inspect audio files and sample submission
- Convert audio to model-ready 16 kHz mono WAV
- Run single-file or batch inference
- Save submission CSV
- Optionally score WER/CER when references are available
- Optionally export timestamps
- Optional long-audio chunking, decoding experiments, normalization, and fallback model hooks

In [ ]:
# ============================================================
# 1. Install required libraries
# ============================================================
import sys
import subprocess

INSTALL_LIBS = True
INSTALL_FFMPEG = True

if INSTALL_FFMPEG:
    try:
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)
    except Exception as e:
        print("apt install skipped or unavailable:", type(e).__name__, e)

if INSTALL_LIBS:
    packages = [
        "nemo_toolkit[asr]",
        "librosa",
        "soundfile",
        "pandas",
        "jiwer",
        "tqdm",
        "seaborn",
        "matplotlib",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("Skipping pip installs.")

In [ ]:
# ============================================================
# 2. Imports
# ============================================================
from pathlib import Path
import os
import re
import gc
import json
import time
import math
import shutil
import zipfile
import subprocess
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import librosa
import soundfile as sf

import matplotlib.pyplot as plt
import seaborn as sns

import torch

try:
    import nemo.collections.asr as nemo_asr
except Exception as e:
    nemo_asr = None
    print("NeMo import failed. Install nemo_toolkit[asr] before model loading.", type(e).__name__, e)

try:
    import jiwer
except Exception:
    jiwer = None

def seed_everything(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# ============================================================
# 3. Configuration placeholders
# ============================================================
# TODO: Edit this cell for each competition.

TASK_TYPE = "asr"
RANDOM_STATE = 42
seed_everything(RANDOM_STATE)

# ----- Colab / data source -----
USE_GOOGLE_DRIVE = True
MOUNT_DRIVE = True
PROJECT_ROOT = Path("/content/drive/MyDrive/asr_preset")

# If using a zip, set DATA_ZIP_PATH. If files are already extracted, set DATA_DIR directly.
DATA_ZIP_PATH = PROJECT_ROOT / "dataset.zip"   # TODO: change this
DATA_DIR = PROJECT_ROOT / "data"
FORCE_EXTRACT = False

# Optional runtime copy is faster in Colab when audio is on Drive.
USE_RUNTIME_COPY = True
RUNTIME_DATA_DIR = Path("/content/asr_data")
FORCE_RUNTIME_RESYNC = False

# ----- File/path placeholders -----
SAMPLE_SUBMISSION_PATH = None      # e.g. DATA_DIR / "sample_submission.csv"
TRAIN_METADATA_PATH = None         # optional CSV with audio path and transcript
TEST_METADATA_PATH = None          # optional CSV with audio path/id
AUDIO_DIRS = []                    # leave empty to auto-detect

# ----- Submission schema -----
ID_COLUMN = None                   # inferred from sample submission if None
AUDIO_COLUMN = None                # e.g. "path", "audio", "filename"
TRANSCRIPT_COLUMN = None           # target/reference column
SUBMISSION_TEXT_COLUMN = None      # inferred from sample submission if None

# ----- Model -----
MODEL_BACKEND = "nemo_typhoon"     # "nemo_typhoon" or custom hook in optional section
MODEL_NAME = "scb10x/typhoon-asr-realtime"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----- Audio preprocessing -----
TARGET_SAMPLE_RATE = 16000
MONO = True
NORMALIZE_AUDIO = True
TRIM_SILENCE = False
TRIM_TOP_DB = 30
OUTPUT_AUDIO_FORMAT = "wav"
PROCESSED_AUDIO_DIR = PROJECT_ROOT / "processed_audio"
REUSE_PROCESSED_AUDIO = True

# ----- Inference -----
RUN_BASELINE_INFERENCE = True
INFERENCE_BATCH_SIZE = 8
RETURN_TIMESTAMPS = False
SAVE_TIMESTAMPS_JSON = False
TIMESTAMPS_JSON_PATH = PROJECT_ROOT / "outputs" / "timestamps.json"

# ----- Long audio chunking -----
USE_CHUNKING = False
CHUNK_SECONDS = 30.0
CHUNK_OVERLAP_SECONDS = 2.0

# ----- Text post-processing -----
LOWERCASE_OUTPUT = False
REMOVE_EXTRA_SPACES = True
REMOVE_PUNCT_FOR_METRIC = False
FALLBACK_TRANSCRIPT = ""

# ----- Output -----
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_PATH = OUTPUT_DIR / "submission.csv"

In [ ]:
# ============================================================
# 4. Dataset extraction and file inspection
# ============================================================
if USE_GOOGLE_DRIVE and MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount skipped or unavailable:", type(e).__name__, e)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

def extract_zip_if_needed(zip_path: Path, out_dir: Path, force: bool = False):
    zip_path = Path(zip_path)
    out_dir = Path(out_dir)
    if not zip_path.exists():
        print("Zip not found. If data is already extracted, this is OK:", zip_path)
        return
    if force and out_dir.exists():
        shutil.rmtree(out_dir)
    if out_dir.exists() and any(out_dir.iterdir()):
        print("Using existing extracted dir:", out_dir)
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    print("Extracted:", zip_path, "->", out_dir)

extract_zip_if_needed(DATA_ZIP_PATH, DATA_DIR, FORCE_EXTRACT)

def sync_runtime_copy(src: Path, dst: Path, force: bool = False):
    if not USE_RUNTIME_COPY:
        return src
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return src
    if force and dst.exists():
        shutil.rmtree(dst)
    if dst.exists() and any(dst.iterdir()):
        print("Using existing runtime copy:", dst)
        return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    if shutil.which("rsync"):
        subprocess.run(["rsync", "-a", "--delete", f"{src}/", f"{dst}/"], check=True)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Runtime copy ready:", dst)
    return dst

ACTIVE_DATA_DIR = sync_runtime_copy(DATA_DIR, RUNTIME_DATA_DIR, FORCE_RUNTIME_RESYNC) if DATA_DIR.exists() else DATA_DIR

def inspect_files(root: Path, max_files: int = 120):
    root = Path(root)
    if not root.exists():
        print("Data dir does not exist yet:", root)
        return []
    files = sorted([p for p in root.rglob("*") if p.is_file()])
    print("file count:", len(files))
    for p in files[:max_files]:
        print(p.relative_to(root), p.stat().st_size)
    if len(files) > max_files:
        print("... truncated")
    return files

all_files = inspect_files(ACTIVE_DATA_DIR)

In [ ]:
# ============================================================
# 5. Auto-detect audio files and metadata
# ============================================================
AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac", ".wma"}

def first_existing(candidates):
    for p in candidates:
        if p is not None and Path(p).exists():
            return Path(p)
    return None

def auto_find_csv(root: Path, patterns):
    hits = []
    for pat in patterns:
        hits.extend(root.rglob(pat))
    return sorted(set(hits), key=lambda p: len(str(p)))[0] if hits else None

SAMPLE_SUBMISSION_PATH = first_existing([SAMPLE_SUBMISSION_PATH]) or auto_find_csv(
    ACTIVE_DATA_DIR,
    ["*sample_submission*.csv", "*submission*.csv"],
)
TRAIN_METADATA_PATH = first_existing([TRAIN_METADATA_PATH]) or auto_find_csv(
    ACTIVE_DATA_DIR,
    ["*train*.csv", "*metadata*.csv", "*transcript*.csv"],
)
TEST_METADATA_PATH = first_existing([TEST_METADATA_PATH]) or auto_find_csv(
    ACTIVE_DATA_DIR,
    ["*test*.csv"],
)

audio_files = sorted([p for p in ACTIVE_DATA_DIR.rglob("*") if p.suffix.lower() in AUDIO_EXTS]) if ACTIVE_DATA_DIR.exists() else []
print("audio files:", len(audio_files))
for p in audio_files[:20]:
    print(" ", p.relative_to(ACTIVE_DATA_DIR))

if not AUDIO_DIRS:
    by_parent = Counter(p.parent for p in audio_files)
    AUDIO_DIRS = [p for p, _ in by_parent.most_common()]

print("SAMPLE_SUBMISSION_PATH:", SAMPLE_SUBMISSION_PATH)
print("TRAIN_METADATA_PATH:", TRAIN_METADATA_PATH)
print("TEST_METADATA_PATH:", TEST_METADATA_PATH)
print("AUDIO_DIRS:", AUDIO_DIRS[:10])

In [ ]:
# ============================================================
# 6. Load data
# ============================================================
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH) if SAMPLE_SUBMISSION_PATH else pd.DataFrame()
train_meta = pd.read_csv(TRAIN_METADATA_PATH) if TRAIN_METADATA_PATH else pd.DataFrame()
test_meta = pd.read_csv(TEST_METADATA_PATH) if TEST_METADATA_PATH else pd.DataFrame()

if not sample_sub.empty:
    ID_COLUMN = ID_COLUMN or sample_sub.columns[0]
    SUBMISSION_TEXT_COLUMN = SUBMISSION_TEXT_COLUMN or sample_sub.columns[-1]

def infer_audio_col(df: pd.DataFrame):
    if df.empty:
        return None
    if AUDIO_COLUMN and AUDIO_COLUMN in df.columns:
        return AUDIO_COLUMN
    candidates = [c for c in df.columns if any(k in c.lower() for k in ["audio", "file", "filename", "path", "wav", "id"])]
    return candidates[0] if candidates else df.columns[0]

def infer_transcript_col(df: pd.DataFrame):
    if df.empty:
        return None
    if TRANSCRIPT_COLUMN and TRANSCRIPT_COLUMN in df.columns:
        return TRANSCRIPT_COLUMN
    candidates = [c for c in df.columns if any(k in c.lower() for k in ["text", "transcript", "sentence", "label", "target"])]
    return candidates[0] if candidates else None

TRAIN_AUDIO_COLUMN = infer_audio_col(train_meta)
TEST_AUDIO_COLUMN = infer_audio_col(test_meta)
TRAIN_TRANSCRIPT_COLUMN = infer_transcript_col(train_meta)

print("sample_sub:", sample_sub.shape)
print("train_meta:", train_meta.shape)
print("test_meta:", test_meta.shape)
print("ID_COLUMN:", ID_COLUMN)
print("SUBMISSION_TEXT_COLUMN:", SUBMISSION_TEXT_COLUMN)
print("TRAIN_AUDIO_COLUMN:", TRAIN_AUDIO_COLUMN)
print("TEST_AUDIO_COLUMN:", TEST_AUDIO_COLUMN)
print("TRAIN_TRANSCRIPT_COLUMN:", TRAIN_TRANSCRIPT_COLUMN)

display(sample_sub.head())
display(train_meta.head())
display(test_meta.head())

In [ ]:
# ============================================================
# 7. Build inference table
# ============================================================
def resolve_audio_path(value, audio_dirs=None):
    audio_dirs = audio_dirs or AUDIO_DIRS
    value = str(value)
    p = Path(value)
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    candidates.append(ACTIVE_DATA_DIR / value)
    for d in audio_dirs:
        d = Path(d)
        candidates.append(d / value)
        candidates.append(d / p.name)
        if p.suffix == "":
            for ext in AUDIO_EXTS:
                candidates.append(d / f"{value}{ext}")
    for c in candidates:
        if c.exists():
            return c
    return candidates[0] if candidates else p

def build_test_table():
    if not test_meta.empty and TEST_AUDIO_COLUMN is not None:
        df = test_meta.copy()
        id_col = ID_COLUMN if ID_COLUMN in df.columns else TEST_AUDIO_COLUMN
        df["_id"] = df[id_col].astype(str)
        df["audio_path"] = df[TEST_AUDIO_COLUMN].map(resolve_audio_path)
        return df

    if not sample_sub.empty:
        df = sample_sub[[ID_COLUMN]].copy()
        df["_id"] = df[ID_COLUMN].astype(str)
        df["audio_path"] = df[ID_COLUMN].map(resolve_audio_path)
        return df

    return pd.DataFrame({
        "_id": [p.stem for p in audio_files],
        "audio_path": audio_files,
    })

test_df = build_test_table()
test_df["audio_exists"] = test_df["audio_path"].map(lambda p: Path(p).exists())
print("test_df:", test_df.shape)
print("audio exists rate:", test_df["audio_exists"].mean() if len(test_df) else None)
display(test_df.head())

In [ ]:
# ============================================================
# 8. Basic audio overview / EDA
# ============================================================
def audio_info(path):
    try:
        info = sf.info(path)
        return {
            "samplerate": info.samplerate,
            "channels": info.channels,
            "duration": float(info.duration),
            "format": info.format,
        }
    except Exception:
        try:
            y, sr = librosa.load(path, sr=None, mono=False, duration=1.0)
            return {"samplerate": sr, "channels": 1 if np.ndim(y) == 1 else y.shape[0], "duration": np.nan, "format": "unknown"}
        except Exception:
            return {"samplerate": np.nan, "channels": np.nan, "duration": np.nan, "format": "unreadable"}

eda_sample = test_df[test_df["audio_exists"]].head(200).copy()
if len(eda_sample):
    info_df = pd.DataFrame([audio_info(p) for p in eda_sample["audio_path"]])
    display(info_df.describe(include="all"))
    if "duration" in info_df:
        plt.figure(figsize=(10, 4))
        sns.histplot(info_df["duration"].dropna(), bins=40)
        plt.title("Audio duration distribution")
        plt.show()
else:
    print("No existing audio files found for EDA.")

In [ ]:
# ============================================================
# 9. Audio preprocessing
# ============================================================
def safe_stem(path: Path):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(path).stem)

def prepare_audio(input_path, output_path=None, target_sr=TARGET_SAMPLE_RATE):
    input_path = Path(input_path)
    if not input_path.exists():
        print("File not found:", input_path)
        return None

    if output_path is None:
        output_path = PROCESSED_AUDIO_DIR / f"{safe_stem(input_path)}_{target_sr}.{OUTPUT_AUDIO_FORMAT}"
    output_path = Path(output_path)

    if REUSE_PROCESSED_AUDIO and output_path.exists():
        return output_path

    try:
        y, sr = librosa.load(input_path, sr=None, mono=MONO)
        if np.ndim(y) > 1 and MONO:
            y = librosa.to_mono(y)
        if TRIM_SILENCE:
            y, _ = librosa.effects.trim(y, top_db=TRIM_TOP_DB)
        if sr != target_sr:
            y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        if NORMALIZE_AUDIO:
            peak = np.max(np.abs(y)) if len(y) else 0
            if peak > 0:
                y = y / peak
        output_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(output_path, y, target_sr)
        return output_path
    except Exception as e:
        print("Audio preprocessing failed:", input_path, type(e).__name__, e)
        return None

def prepare_audio_batch(paths):
    processed = []
    for p in tqdm(paths):
        processed.append(prepare_audio(p))
    return processed

In [ ]:
# ============================================================
# 10. Model loading
# ============================================================
def load_asr_model():
    if MODEL_BACKEND != "nemo_typhoon":
        raise ValueError("Only nemo_typhoon is implemented by default. See optional custom backend cell.")
    if nemo_asr is None:
        raise ImportError("nemo.collections.asr is unavailable. Run install/import cells first.")
    print("Using device:", DEVICE)
    print("Loading ASR model:", MODEL_NAME)
    model = nemo_asr.models.ASRModel.from_pretrained(
        model_name=MODEL_NAME,
        map_location=DEVICE,
    )
    return model

asr_model = load_asr_model() if RUN_BASELINE_INFERENCE else None

In [ ]:
# ============================================================
# 11. Inference helpers
# ============================================================
def extract_text_from_transcription(obj):
    if obj is None:
        return FALLBACK_TRANSCRIPT
    if isinstance(obj, str):
        return obj
    if hasattr(obj, "text"):
        return obj.text
    if isinstance(obj, dict):
        return obj.get("text", FALLBACK_TRANSCRIPT)
    return str(obj)

def normalize_transcript(text: str, for_metric: bool = False):
    text = str(text)
    if LOWERCASE_OUTPUT:
        text = text.lower()
    if REMOVE_EXTRA_SPACES:
        text = re.sub(r"\s+", " ", text).strip()
    if for_metric and REMOVE_PUNCT_FOR_METRIC:
        text = re.sub(r"[^\w\s\u0E00-\u0E7F]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
    return text

def transcribe_files(model, audio_paths, batch_size=INFERENCE_BATCH_SIZE, timestamps=RETURN_TIMESTAMPS):
    rows = []
    start_all = time.time()
    for start in tqdm(range(0, len(audio_paths), batch_size)):
        batch = audio_paths[start:start + batch_size]
        batch = [str(p) for p in batch if p is not None and Path(p).exists()]
        if not batch:
            continue
        t0 = time.time()
        outputs = model.transcribe(audio=batch, timestamps=timestamps)
        batch_time = time.time() - t0
        for path, out in zip(batch, outputs):
            try:
                duration = sf.info(path).duration
            except Exception:
                duration = np.nan
            rows.append({
                "audio_path": path,
                "transcript": normalize_transcript(extract_text_from_transcription(out)),
                "duration": duration,
                "batch_seconds": batch_time,
                "rtf_estimate": batch_time / max(duration, 1e-9) if np.isfinite(duration) else np.nan,
                "raw_output": out,
            })
    print("total inference seconds:", round(time.time() - start_all, 2))
    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 12. Baseline batch inference
# ============================================================
pred_df = pd.DataFrame()
timestamp_payload = {}

if RUN_BASELINE_INFERENCE and asr_model is not None:
    existing = test_df[test_df["audio_exists"]].reset_index(drop=True)
    processed_paths = prepare_audio_batch(existing["audio_path"].tolist())
    existing["processed_audio_path"] = processed_paths
    existing = existing[existing["processed_audio_path"].notna()].reset_index(drop=True)

    pred_df = transcribe_files(
        asr_model,
        existing["processed_audio_path"].tolist(),
        batch_size=INFERENCE_BATCH_SIZE,
        timestamps=RETURN_TIMESTAMPS,
    )
    pred_df["_id"] = existing["_id"].values[:len(pred_df)]
    display(pred_df.head())

    if SAVE_TIMESTAMPS_JSON and RETURN_TIMESTAMPS:
        for row in pred_df.itertuples(index=False):
            raw = getattr(row, "raw_output")
            if hasattr(raw, "timestamp"):
                timestamp_payload[row._id] = raw.timestamp
        TIMESTAMPS_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
        TIMESTAMPS_JSON_PATH.write_text(json.dumps(timestamp_payload, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Saved timestamps:", TIMESTAMPS_JSON_PATH)
else:
    print("Baseline inference disabled.")

In [ ]:
# ============================================================
# 13. Validation and metric calculation
# ============================================================
def compute_asr_metrics(preds, refs):
    preds = [normalize_transcript(x, for_metric=True) for x in preds]
    refs = [normalize_transcript(x, for_metric=True) for x in refs]
    if jiwer is None:
        print("jiwer unavailable; install jiwer for WER/CER.")
        return {}
    return {
        "wer": jiwer.wer(refs, preds),
        "cer": jiwer.cer(refs, preds),
    }

metric_scores = {}
if not train_meta.empty and TRAIN_TRANSCRIPT_COLUMN and TRAIN_AUDIO_COLUMN:
    # Optional quick validation on rows that overlap with pred_df IDs, if any.
    print("Reference transcript column detected:", TRAIN_TRANSCRIPT_COLUMN)
else:
    print("No reference transcripts configured for validation scoring.")

# If you have validation predictions and references, call:
# metric_scores = compute_asr_metrics(pred_list, ref_list)

In [ ]:
# ============================================================
# 14. Submission file generation
# ============================================================
if not pred_df.empty:
    if sample_sub.empty:
        submission = pred_df[["_id", "transcript"]].rename(columns={"_id": ID_COLUMN or "id", "transcript": SUBMISSION_TEXT_COLUMN or "text"})
    else:
        submission = sample_sub.copy()
        pred_map = dict(zip(pred_df["_id"].astype(str), pred_df["transcript"]))
        submission[SUBMISSION_TEXT_COLUMN] = submission[ID_COLUMN].astype(str).map(pred_map).fillna(FALLBACK_TRANSCRIPT)

    submission[SUBMISSION_TEXT_COLUMN] = submission[SUBMISSION_TEXT_COLUMN].map(normalize_transcript)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False)
    print("Saved:", OUTPUT_PATH)
    display(submission.head())
    print("rows:", len(submission), "blank transcripts:", int((submission[SUBMISSION_TEXT_COLUMN].astype(str).str.strip() == "").sum()))
else:
    print("No predictions yet; run inference first.")

## Optional Improvement Ideas

Each optional section is guarded by a flag so the baseline remains quick and easy to debug.

In [ ]:
# ============================================================
# Optional A. Long-audio chunking
# Use when files are too long or model memory/latency is unstable.
# Tradeoff: faster/safer inference, but stitching can duplicate or drop words near boundaries.
# ============================================================
RUN_CHUNKING_EXPERIMENT = USE_CHUNKING

def split_audio_to_chunks(path, chunk_seconds=CHUNK_SECONDS, overlap_seconds=CHUNK_OVERLAP_SECONDS, target_sr=TARGET_SAMPLE_RATE):
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    chunk_len = int(chunk_seconds * target_sr)
    overlap_len = int(overlap_seconds * target_sr)
    step = max(1, chunk_len - overlap_len)
    chunks = []
    out_dir = PROCESSED_AUDIO_DIR / f"chunks_{safe_stem(path)}"
    out_dir.mkdir(parents=True, exist_ok=True)
    for i, start in enumerate(range(0, len(y), step)):
        end = min(len(y), start + chunk_len)
        chunk = y[start:end]
        if len(chunk) < int(0.5 * target_sr):
            continue
        out_path = out_dir / f"chunk_{i:04d}.wav"
        sf.write(out_path, chunk, target_sr)
        chunks.append({"chunk_path": out_path, "start_sec": start / target_sr, "end_sec": end / target_sr})
        if end >= len(y):
            break
    return pd.DataFrame(chunks)

if RUN_CHUNKING_EXPERIMENT and asr_model is not None and len(test_df):
    one_path = test_df[test_df["audio_exists"]]["audio_path"].iloc[0]
    chunks = split_audio_to_chunks(one_path)
    chunk_preds = transcribe_files(asr_model, chunks["chunk_path"].tolist(), batch_size=INFERENCE_BATCH_SIZE)
    stitched = " ".join(chunk_preds["transcript"].tolist())
    print(normalize_transcript(stitched))
else:
    print("Chunking experiment disabled.")

In [ ]:
# ============================================================
# Optional B. Text normalization experiments
# Use when the metric ignores punctuation/case or Thai spacing conventions.
# Tradeoff: can improve metrics, but may hurt if exact formatting is scored.
# ============================================================
RUN_NORMALIZATION_GRID = False

if RUN_NORMALIZATION_GRID:
    assert "pred_list" in globals() and "ref_list" in globals(), "Define pred_list and ref_list first."
    def normalize_for_grid(text: str, lower: bool, remove_punct: bool):
        text = str(text)
        if lower:
            text = text.lower()
        text = re.sub(r"\s+", " ", text).strip()
        if remove_punct:
            text = re.sub(r"[^\w\s\u0E00-\u0E7F]", "", text)
            text = re.sub(r"\s+", " ", text).strip()
        return text
    settings = [
        {"lower": False, "punct": False},
        {"lower": True, "punct": False},
        {"lower": False, "punct": True},
        {"lower": True, "punct": True},
    ]
    rows = []
    for s in settings:
        preds_norm = [normalize_for_grid(x, s["lower"], s["punct"]) for x in pred_list]
        refs_norm = [normalize_for_grid(x, s["lower"], s["punct"]) for x in ref_list]
        rows.append({**s, "wer": jiwer.wer(refs_norm, preds_norm), "cer": jiwer.cer(refs_norm, preds_norm)})
    display(pd.DataFrame(rows).sort_values("wer"))
else:
    print("Normalization grid disabled.")

In [ ]:
# ============================================================
# Optional C. Timestamp export
# Use for alignment tasks or subtitle-style deliverables.
# Tradeoff: slower inference and larger output files.
# ============================================================
RUN_TIMESTAMP_EXPORT = False

if RUN_TIMESTAMP_EXPORT:
    RETURN_TIMESTAMPS = True
    SAVE_TIMESTAMPS_JSON = True
    # Re-run inference cell after setting these flags.
    print("Set RETURN_TIMESTAMPS=True and rerun inference.")
else:
    print("Timestamp export disabled.")

In [ ]:
# ============================================================
# Optional D. Alternative backend hook
# Use when the competition language/model is not supported by Typhoon ASR.
# Tradeoff: more setup, but lets the notebook support Whisper, wav2vec2, etc.
# ============================================================
RUN_CUSTOM_BACKEND_EXAMPLE = False

if RUN_CUSTOM_BACKEND_EXAMPLE:
    # TODO: implement another ASR model here, for example Hugging Face Whisper.
    # from transformers import pipeline
    # whisper = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=0 if torch.cuda.is_available() else -1)
    # text = whisper(str(audio_path))["text"]
    raise NotImplementedError("Add your custom backend here.")
else:
    print("Custom backend example disabled.")

In [ ]:
# ============================================================
# Optional E. Error analysis
# Use when references are available.
# Tradeoff: manual inspection time, but often reveals audio quality or normalization issues.
# ============================================================
RUN_ERROR_ANALYSIS = False

if RUN_ERROR_ANALYSIS:
    assert "error_df" in globals(), "Create error_df with columns: id, prediction, reference, audio_path."
    if jiwer is None:
        print("Install jiwer for detailed transformations.")
    error_df["pred_len"] = error_df["prediction"].astype(str).str.len()
    error_df["ref_len"] = error_df["reference"].astype(str).str.len()
    error_df["len_gap"] = (error_df["pred_len"] - error_df["ref_len"]).abs()
    display(error_df.sort_values("len_gap", ascending=False).head(30))
else:
    print("Error analysis disabled.")

## Debug Checklist

Before submitting:

1. Confirm `DATA_DIR`, `AUDIO_DIRS`, and sample submission columns.
2. Confirm `audio_exists` is close to `1.0`.
3. Run one or two files first before full batch inference.
4. Check output text manually for language, spacing, and punctuation.
5. Confirm submission row count matches the sample submission.
6. Confirm no unintended blank transcripts.
7. Save the model name and config used for each submission.

If something fails:

- If NeMo install fails, restart runtime and rerun install/import cells.
- If audio loading fails, make sure `ffmpeg` is installed and file paths are correct.
- If GPU memory is tight, lower `INFERENCE_BATCH_SIZE`.
- If long files fail, enable `USE_CHUNKING`.
- If metric is poor, tune text normalization and inspect common error cases.